Buat SparkSession baru:

In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

A. Membaca dan Eksplorasi Awal 

Baca data langsung dari HDFS:

In [12]:
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

df.printSchema()

print("Jumlah baris:", df.count())

df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

B. Menangani Data Kosong

In [14]:
from pyspark.sql.functions import col, sum as spark_sum

df.select(
    spark_sum(
        col("rating").isNull().cast("int")
    ).alias("jumlah_rating_kosong")
).show()

+--------------------+
|jumlah_rating_kosong|
+--------------------+
|                 204|
+--------------------+



In [15]:
rata_rating = df.selectExpr(
    "avg(rating) as rata_rating"
).first()["rata_rating"]

df = df.na.fill({
    "rating": rata_rating
})

df.select(
    spark_sum(
        col("rating").isNull().cast("int")
    ).alias("jumlah_rating_kosong")
).show()

+--------------------+
|jumlah_rating_kosong|
+--------------------+
|                   0|
+--------------------+



C. Transformasi Data

In [16]:
from pyspark.sql.functions import when

df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df = df.withColumn(
    "tier_transaksi",
    when(
        col("total_pendapatan") > 500000,
        "Besar"
    ).otherwise("Kecil")
)

df.select(
    "order_id",
    "unit_terjual",
    "harga_satuan",
    "total_pendapatan",
    "tier_transaksi"
).show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



D. Analisis dengan GroupBy

In [17]:
hasil_kategori = df.groupBy("kategori") \
    .sum("total_pendapatan") \
    .withColumnRenamed(
        "sum(total_pendapatan)",
        "total_pendapatan"
    ) \
    .orderBy(
        col("total_pendapatan").desc()
    )

hasil_kategori.show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



In [18]:
hasil_kota = df.filter(
    col("tier_transaksi") == "Besar"
).groupBy("kota") \
 .count() \
 .orderBy(
     col("count").desc()
 )

hasil_kota.show()

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



In [19]:
hasil_rating = df.groupBy(
    "metode_pembayaran"
).avg("rating") \
 .withColumnRenamed(
     "avg(rating)",
     "rata_rata_rating"
 ) \
 .orderBy(
     col("rata_rata_rating").desc()
 )

hasil_rating.show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD| 4.167310656870009|
|    Transfer Bank|   4.1592349097265|
|         E-Wallet| 4.137728643216084|
|     Kartu Kredit|4.1179474608816475|
+-----------------+------------------+



E. Menyimpan Hasil ke HDFS

In [21]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan"

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

In [22]:
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_olahan

Found 2 items
-rw-r--r--   3 pathan supergroup          0 2026-09-24 00:33 /user/mahasiswa/tugas4/hasil_olahan/_SUCCESS
-rw-r--r--   3 pathan supergroup     100356 2026-09-24 00:33 /user/mahasiswa/tugas4/hasil_olahan/part-00000-49adf38a-ce92-4937-b17d-47c734488317-c000.csv


In [23]:
!hdfs dfs -cat /user/mahasiswa/tugas4/hasil_olahan/part-*.csv | head

order_id,tanggal,kategori,kota,unit_terjual,harga_satuan,metode_pembayaran,rating,total_pendapatan,tier_transaksi
ORD-3000,2026-09-02T00:00:00.000+07:00,Rumah Tangga,Yogyakarta,3,90000,COD,4.0,270000,Kecil
ORD-3001,2026-09-04T00:00:00.000+07:00,Makanan & Minuman,Solo,3,200000,E-Wallet,5.0,600000,Besar
ORD-3002,2026-09-26T00:00:00.000+07:00,Kesehatan & Kecantikan,Semarang,8,60000,E-Wallet,3.0,480000,Kecil
ORD-3003,2026-09-09T00:00:00.000+07:00,Makanan & Minuman,Semarang,6,350000,Transfer Bank,4.0,2100000,Besar
ORD-3004,2026-09-10T00:00:00.000+07:00,Rumah Tangga,Yogyakarta,10,60000,E-Wallet,4.0,600000,Besar
ORD-3005,2026-09-09T00:00:00.000+07:00,Fashion,Purworejo,5,20000,E-Wallet,4.0,100000,Kecil
ORD-3006,2026-09-19T00:00:00.000+07:00,Makanan & Minuman,Yogyakarta,2,20000,COD,5.0,40000,Kecil
ORD-3007,2026-09-05T00:00:00.000+07:00,Makanan & Minuman,Magelang,8,90000,Transfer Bank,4.1457286432160805,720000,Besar
ORD-3008,2026-09-06T00:00:00.000+07:00,Fashion,Semarang,7,20000,Kartu Kredit,5.0

In [24]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
